# 01 — Data Cleaning & Preprocessing
**PSAJ Statistika | Kelompok: Fajri, Aufa, Andre**

Notebook ini:
1. Load **Dataset X** (Pendidikan SMA+) & **Dataset Y** (TPT) untuk tahun 2018–2025
2. Filter ke level 38 **Provinsi** saja
3. Merge X dan Y per tahun → `merged_{year}.csv`
4. Buat `merged_all_years.csv` (multi-tahun, untuk Backtesting)

---
**Output:**
- `outputs/merged_2025.csv` — data cross-section 2025 (untuk Fase 2 & 3 PSAJ wajib)
- `outputs/merged_all_years.csv` — data panel 2018–2025 (untuk Fase 4 Backtesting)

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/tables', exist_ok=True)

YEARS = list(range(2018, 2026))  # 2018 s/d 2025
DIR_X = 'datasets/Dataset X'
DIR_Y = 'datasets/Dataset Y (Tingkat Pengangguran Terbuka)'

print('Setup selesai. Tahun yang akan diproses:', YEARS)

Setup selesai. Tahun yang akan diproses: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Fungsi Bantu

In [2]:
# Mapping normalisasi nama provinsi (Dataset X pakai nama panjang, Y pakai singkat)
NAMA_MAPPING = {
    'D I YOGYAKARTA'       : 'DI YOGYAKARTA',
    'KEPULAUAN RIAU'       : 'KEP. RIAU',
    'KEP. BANGKA BELITUNG' : 'KEP. BANGKA BELITUNG',
    'BANGKA BELITUNG'      : 'KEP. BANGKA BELITUNG',
    'BANGKA BELITUNG ISLANDS': 'KEP. BANGKA BELITUNG',
}

def normalize(name):
    name = str(name).strip().upper()
    return NAMA_MAPPING.get(name, name)

def is_province_row(name):
    """True jika baris adalah level provinsi (ALL CAPS, bukan INDONESIA)"""
    if pd.isna(name): return False
    name = str(name).strip()
    if name in ('INDONESIA', ''): return False
    return name == name.upper() and len(name) > 2 and not name[0].isdigit()

def load_x(year):
    """Load Dataset X untuk satu tahun, return DataFrame level provinsi."""
    path = f"{DIR_X}/Persentase penduduk usia 25 Tahun keatas dengan pendidikan SMA ke atas menurut jenis kelamin, {year}.csv"
    df = pd.read_csv(path, skiprows=3, header=None, names=['Provinsi', 'Laki_laki', 'Perempuan'], encoding='utf-8-sig')
    df = df[df['Provinsi'].apply(is_province_row)].copy().reset_index(drop=True)
    df['Laki_laki'] = pd.to_numeric(df['Laki_laki'], errors='coerce')
    df['Perempuan'] = pd.to_numeric(df['Perempuan'], errors='coerce')
    df['SMA_Plus_Pct'] = (df['Laki_laki'] + df['Perempuan']) / 2
    df['Provinsi_key'] = df['Provinsi'].apply(normalize)
    df['Year'] = year
    return df[['Year', 'Provinsi_key', 'Laki_laki', 'Perempuan', 'SMA_Plus_Pct']].rename(columns={'Provinsi_key': 'Provinsi'})

def load_y(year):
    """Load Dataset Y untuk satu tahun, ambil kolom Agustus."""
    path = f"{DIR_Y}/Tingkat Pengangguran Terbuka Menurut Provinsi, {year}.csv"
    df = pd.read_csv(path, skiprows=3, header=None, names=['Provinsi', 'Februari', 'Agustus', 'Tahunan'], encoding='utf-8-sig')
    df = df[df['Provinsi'].notna() & (~df['Provinsi'].isin(['INDONESIA']))].copy().reset_index(drop=True)
    df['Agustus'] = pd.to_numeric(df['Agustus'], errors='coerce')
    df['Provinsi_key'] = df['Provinsi'].apply(normalize)
    df['Year'] = year
    return df[['Year', 'Provinsi_key', 'Agustus']].rename(columns={'Provinsi_key': 'Provinsi', 'Agustus': 'TPT_Pct'})

print('Fungsi bantu siap.')

Fungsi bantu siap.


## Load & Merge Per Tahun

In [3]:
all_frames = []

for year in YEARS:
    df_x = load_x(year)
    df_y = load_y(year)
    
    df_merged = pd.merge(df_x, df_y, on=['Year', 'Provinsi'], how='inner')
    df_clean  = df_merged.dropna(subset=['SMA_Plus_Pct', 'TPT_Pct'])
    
    all_frames.append(df_clean)
    
    # Cek jumlah
    if len(df_clean) != 38:
        x_keys = set(df_x['Provinsi'])
        y_keys = set(df_y['Provinsi'])
        missing = x_keys.symmetric_difference(y_keys)
        print(f'{year}: ⚠️  {len(df_clean)} baris (bukan 38). Provinsi tidak match: {missing}')
    else:
        print(f'{year}: ✅ {len(df_clean)} provinsi')

df_all = pd.concat(all_frames, ignore_index=True)
print(f'\nTotal baris semua tahun: {len(df_all)}')

2018: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2019: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2020: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2021: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2022: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2023: ⚠️  34 baris (bukan 38). Provinsi tidak match: set()
2024: ✅ 38 provinsi
2025: ✅ 38 provinsi

Total baris semua tahun: 280


In [4]:
# Simpan data 2025 (cross-section untuk PSAJ wajib)
df_2025 = df_all[df_all['Year'] == 2025].reset_index(drop=True)
df_2025.to_csv('outputs/merged_2025.csv', index=False)
print(f'Tersimpan: outputs/merged_2025.csv ({len(df_2025)} baris)')

# Simpan semua tahun (panel data untuk Backtesting)
df_all.to_csv('outputs/merged_all_years.csv', index=False)
print(f'Tersimpan: outputs/merged_all_years.csv ({len(df_all)} baris)')

# Preview
print('\n=== Preview Data 2025 (5 baris pertama) ===')
print(df_2025.head().to_string())

print('\n=== Summary Statistics 2025 ===')
print(df_2025[['SMA_Plus_Pct', 'TPT_Pct']].describe().round(4))

Tersimpan: outputs/merged_2025.csv (38 baris)
Tersimpan: outputs/merged_all_years.csv (280 baris)

=== Preview Data 2025 (5 baris pertama) ===
   Year        Provinsi  Laki_laki  Perempuan  SMA_Plus_Pct  TPT_Pct
0  2025            ACEH      53.12      49.76        51.440     5.64
1  2025  SUMATERA UTARA      55.64      52.01        53.825     5.32
2  2025  SUMATERA BARAT      46.66      50.12        48.390     5.62
3  2025            RIAU      46.88      44.22        45.550     4.16
4  2025           JAMBI      42.25      36.77        39.510     4.26

=== Summary Statistics 2025 ===
       SMA_Plus_Pct  TPT_Pct
count       38.0000  38.0000
mean        43.6575   4.4682
std         10.3869   1.3813
min         16.6600   1.4900
25%         36.4612   3.5000
50%         42.2575   4.2100
75%         50.3512   5.5450
max         68.2950   6.9600


In [5]:
# Tabel ringkasan rata-rata nasional per tahun (untuk grafik tren di Bab 4.4)
tren = df_all.groupby('Year')[['SMA_Plus_Pct', 'TPT_Pct']].mean().round(4)
tren.columns = ['Rata-rata SMA+%', 'Rata-rata TPT%']
tren.to_csv('outputs/tables/tren_nasional_2018_2025.csv')
print('=== Tren Rata-rata Nasional 2018-2025 ===')
print(tren.to_string())

=== Tren Rata-rata Nasional 2018-2025 ===
      Rata-rata SMA+%  Rata-rata TPT%
Year                                 
2018          37.9657          4.8032
2019          39.4528          4.7124
2020          40.4932          6.0335
2021          40.3806          5.4921
2022          41.6803          4.9662
2023          42.2054          4.6138
2024          42.8841          4.3797
2025          43.6575          4.4682
